# CNN Crop and Weed Classification

**Assigned roll numbers:** 41-45  
**Task:** Binary image classification with TensorFlow and Keras

This notebook downloads the V2 Plant Seedlings Dataset, maps its 12 plant species to **Crop** or **Weed**, creates reproducible train/validation/test splits, trains a custom CNN, and saves the graphs and metrics required for the report and presentation.

Use a Google Colab GPU. Run the cells from top to bottom and do not report results until the final evaluation cells finish.


## 1 Setup

The only extra Colab dependency is KaggleHub, which downloads the public dataset. TensorFlow, pandas, scikit-learn, matplotlib, and seaborn are normally preinstalled in Colab.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("kagglehub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

import json
import random
import re
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
LOCAL_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "plant_seedlings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Project root:", PROJECT_ROOT)


## 2 Download and locate the dataset

The dataset contains 5,539 RGB seedling images in 12 species folders. If `data/raw/plant_seedlings` already contains those folders, the notebook uses the local copy. Otherwise it downloads the public Kaggle dataset `vbookshelf/v2-plant-seedlings-dataset`.


In [ ]:
DATASET_HANDLE = "vbookshelf/v2-plant-seedlings-dataset"

if LOCAL_DATA_DIR.exists() and any(LOCAL_DATA_DIR.rglob("*.png")):
    DATASET_ROOT = LOCAL_DATA_DIR
    print("Using local dataset:", DATASET_ROOT)
else:
    DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Downloaded dataset:", DATASET_ROOT)


## 3 Convert 12 species into Crop and Weed

Crop species are Maize, Common wheat, and Sugar beet. All other species are weeds. Label `0` means Crop and label `1` means Weed, so the sigmoid output is the estimated probability of Weed.


In [ ]:
CROP_SPECIES = {"maize", "common_wheat", "sugar_beet"}
EXPECTED_SPECIES = {
    "black_grass", "charlock", "cleavers", "common_chickweed",
    "common_wheat", "fat_hen", "loose_silkybent", "maize",
    "scentless_mayweed", "shepherds_purse",
    "small_flowered_cranesbill", "sugar_beet",
}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

def normalize_species(name):
    normalized = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    aliases = {
        "blackgrass": "black_grass",
        "commonwheat": "common_wheat",
        "fathen": "fat_hen",
        "loose_silky_bent": "loose_silkybent",
        "loosesilkybent": "loose_silkybent",
        "scentlessmayweed": "scentless_mayweed",
        "shepherd_s_purse": "shepherds_purse",
        "shepherdspurse": "shepherds_purse",
        "smallflowered_cranesbill": "small_flowered_cranesbill",
        "sugarbeet": "sugar_beet",
    }
    return aliases.get(normalized, normalized)

rows = []
for path in DATASET_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        species = normalize_species(path.parent.name)
        if species in EXPECTED_SPECIES:
            binary_name = "Crop" if species in CROP_SPECIES else "Weed"
            rows.append({
                "path": str(path),
                "species": species,
                "label_name": binary_name,
                "label": 0 if binary_name == "Crop" else 1,
            })

images = pd.DataFrame(rows).drop_duplicates(subset="path").reset_index(drop=True)
found_species = set(images["species"].unique())

assert len(images) >= 5_000, f"Expected about 5,539 images, found {len(images)}"
assert found_species == EXPECTED_SPECIES, (
    f"Species mismatch. Missing={EXPECTED_SPECIES - found_species}; "
    f"unexpected={found_species - EXPECTED_SPECIES}"
)

print("Images:", len(images))
display(images.head())
display(images.groupby(["label_name", "species"]).size().rename("images").to_frame())


## 4 Explore the dataset


In [ ]:
sns.set_theme(style="whitegrid")

class_counts = images["label_name"].value_counts().reindex(["Crop", "Weed"])
ax = class_counts.plot(kind="bar", color=["#2E7D32", "#C45A31"], figsize=(6, 4))
ax.set_title("Crop and Weed Image Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=0)
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=200)
plt.show()

samples = images.groupby("label_name", group_keys=False).sample(n=4, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, row in zip(axes.flat, samples.itertuples()):
    image = tf.keras.utils.load_img(row.path)
    ax.imshow(image)
    ax.set_title(f"{row.label_name}\n{row.species.replace('_', ' ').title()}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_images.png", dpi=200)
plt.show()


## 5 Create reproducible splits

Stratification preserves the Crop/Weed proportion. The 70/15/15 split is performed before augmentation. The assertions prove that no file appears in more than one split.


In [ ]:
train_df, remaining_df = train_test_split(
    images,
    test_size=0.30,
    random_state=SEED,
    stratify=images["label"],
)
val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=SEED,
    stratify=remaining_df["label"],
)

train_paths = set(train_df["path"])
val_paths = set(val_df["path"])
test_paths = set(test_df["path"])
assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)
assert len(train_df) + len(val_df) + len(test_df) == len(images)

split_table = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="validation"),
    test_df.assign(split="test"),
], ignore_index=True)
split_table.to_csv(OUTPUT_DIR / "dataset_splits.csv", index=False)
display(pd.crosstab(split_table["split"], split_table["label_name"], margins=True))


## 6 Build the TensorFlow input pipeline


In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["label"].values)
    )
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].to_numpy(),
)
CLASS_WEIGHTS = {0: float(weights[0]), 1: float(weights[1])}
print("Class weights:", CLASS_WEIGHTS)


## 7 Build the CNN

Augmentation is part of the model and is active only during training. Three convolution blocks learn progressively richer visual features. Dropout reduces overfitting. The final sigmoid neuron returns Weed probability.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.08, seed=SEED),
    tf.keras.layers.RandomZoom(0.10, seed=SEED),
    tf.keras.layers.RandomTranslation(0.05, 0.05, seed=SEED),
], name="augmentation")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    augmentation,
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.50, seed=SEED),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="crop_weed_cnn")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)
model.summary()


## 8 Train and save the best model


In [ ]:
BEST_MODEL_PATH = MODEL_DIR / "best_crop_weed_cnn.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks,
)


## 9 Plot accuracy and loss


In [ ]:
epochs_run = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(7, 5))
plt.plot(epochs_run, history.history["accuracy"], marker="o", label="Training")
plt.plot(epochs_run, history.history["val_accuracy"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_graph.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(epochs_run, history.history["loss"], marker="o", label="Training")
plt.plot(epochs_run, history.history["val_loss"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("CNN Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loss_graph.png", dpi=200)
plt.show()


## 10 Final test evaluation

The test set is used here for the first time. Because label 1 is Weed, Keras precision and recall describe the Weed class. The classification report gives separate values for Crop and Weed.


In [ ]:
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)
evaluation = best_model.evaluate(test_ds, return_dict=True, verbose=1)

probabilities = best_model.predict(test_ds, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype(int)
actual = test_df["label"].to_numpy()

report = classification_report(
    actual,
    predictions,
    target_names=["Crop", "Weed"],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")

summary = {
    "random_seed": SEED,
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "epochs_completed": int(len(history.history["loss"])),
    "test_loss": float(evaluation["loss"]),
    "test_accuracy": float(evaluation["accuracy"]),
    "test_auc": float(evaluation["auc"]),
    "crop_precision": float(report["Crop"]["precision"]),
    "crop_recall": float(report["Crop"]["recall"]),
    "crop_f1": float(report["Crop"]["f1-score"]),
    "weed_precision": float(report["Weed"]["precision"]),
    "weed_recall": float(report["Weed"]["recall"]),
    "weed_f1": float(report["Weed"]["f1-score"]),
}
(OUTPUT_DIR / "results_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print(json.dumps(summary, indent=2))
display(report_df)


## 11 Confusion matrix


In [ ]:
matrix = confusion_matrix(actual, predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Crop", "Weed"],
    yticklabels=["Crop", "Weed"],
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Crop and Weed Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=200)
plt.show()


## 12 Sample predictions and error analysis


In [ ]:
results = test_df.reset_index(drop=True).copy()
results["probability_weed"] = probabilities
results["prediction"] = predictions
results["predicted_name"] = np.where(predictions == 1, "Weed", "Crop")
results["correct"] = results["label"].to_numpy() == predictions
results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

def plot_examples(frame, filename, title, count=12):
    chosen = frame.head(count)
    if chosen.empty:
        print(f"No images available for {title}")
        return
    columns = 4
    rows = int(np.ceil(len(chosen) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, row in zip(axes, chosen.itertuples()):
        ax.imshow(tf.keras.utils.load_img(row.path))
        ax.set_title(
            f"Actual: {row.label_name}\nPredicted: {row.predicted_name}\n"
            f"P(Weed): {row.probability_weed:.2f}",
            color="green" if row.correct else "red",
        )
        ax.axis("off")
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()

plot_examples(
    results.sample(n=min(12, len(results)), random_state=SEED),
    "sample_predictions.png",
    "Sample Test Predictions",
)
plot_examples(
    results.loc[~results["correct"]].sort_values("probability_weed"),
    "misclassified_examples.png",
    "Misclassified Test Images",
)


## 13 Predict one new image

Change `IMAGE_TO_PREDICT` to any PNG or JPG. The result is a model prediction, not agricultural advice; field conditions may differ from this controlled dataset.


In [ ]:
def predict_one(image_path):
    image = tf.keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    array = tf.keras.utils.img_to_array(image) / 255.0
    probability_weed = float(best_model.predict(array[None, ...], verbose=0)[0, 0])
    label = "Weed" if probability_weed >= 0.5 else "Crop"
    confidence = probability_weed if label == "Weed" else 1.0 - probability_weed
    return {"prediction": label, "confidence": confidence, "probability_weed": probability_weed}

IMAGE_TO_PREDICT = test_df.iloc[0]["path"]
print(predict_one(IMAGE_TO_PREDICT))


## 14 What to copy into the report and PPT

Use `results_summary.json` for all numbers. Use `accuracy_graph.png`, `loss_graph.png`, `confusion_matrix.png`, `sample_predictions.png`, and `misclassified_examples.png` as figures. State that the system classifies isolated seedling images and has not been validated on crowded real-field scenes.
